# Laura++ / Anisovich–Sarantsev K-matrix validation

This notebook validates the five-pole, five-channel $\pi\pi$ S-wave K-matrix implementation.

The scattering parameters are fixed to the standard Anisovich–Sarantsev set used by Laura++:

- five bare poles;
- five channels: $\pi\pi$, $K\bar K$, $4\pi$, $\eta\eta$, $\eta\eta'$;
- slowly-varying scattering term;
- Adler-zero factor;
- P-vector production formalism.

The production coefficients used below are **illustrative**, chosen only to make all pieces visible. They are not a fit to a particular data set. The notebook checks the pure K-matrix/P-vector amplitude, the complete symmetrized $D^+\to\pi^-\pi^+\pi^+$ component, deterministic Dalitz-grid normalization, and a 100k-event toy MC.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel,
    DecayModel,
    KMatrix,
    RealImag,
    Resonance,
    ResonanceContext,
    enable_x64,
    weighted_resample,
)

enable_x64()


## 1. Channel and illustrative production vector


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
mpi_minus, mpi_plus, _ = channel.daughter_masses

kmatrix = KMatrix(
    betas=(
        RealImag(1.00, 0.00),
        RealImag(0.35, -0.20),
        RealImag(-0.15, 0.25),
        RealImag(0.10, 0.05),
        RealImag(0.05, -0.08),
    ),
    f_prod=(
        RealImag(0.20, 0.10),
        RealImag(-0.08, 0.04),
        RealImag(0.03, -0.02),
        RealImag(0.00, 0.00),
        RealImag(0.00, 0.00),
    ),
)

print("D+ mass =", channel.parent_mass, "GeV")
print("daughter masses =", channel.daughter_masses, "GeV")


## 2. Five-channel phase-space factors

Closed two-body channels are analytically continued below threshold. The $4\pi$ channel follows the Anisovich–Sarantsev approximation used by Laura++.


In [ ]:
m = jnp.linspace(2.0 * mpi_plus + 1e-4, 1.72, 2200)
rho = np.asarray(kmatrix.phase_space(m))
labels = [r"$\pi\pi$", r"$K\bar K$", r"$4\pi$", r"$\eta\eta$", r"$\eta\eta'$"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
for i, label in enumerate(labels):
    axes[0].plot(np.asarray(m), rho[:, i].real, label=label)
    axes[1].plot(np.asarray(m), rho[:, i].imag, label=label)
axes[0].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel=r"Re $\rho_i$", title="K-matrix phase space — real parts")
axes[1].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel=r"Im $\rho_i$", title="K-matrix phase space — imaginary parts")
axes[0].legend(ncol=2)
axes[1].legend(ncol=2)
plt.show()


## 3. Scattering matrix $K(s)$

The matrix must be real and symmetric above/below thresholds because the channel continuation enters through $\rho$, not through $K$. The plots below show the five entries coupling the $\pi\pi$ channel to the five scattering channels.


In [ ]:
K = np.asarray(kmatrix.scattering_matrix(m))
print("max symmetry residual =", np.max(np.abs(K - np.swapaxes(K, -1, -2))))

fig, ax = plt.subplots(figsize=(9, 5.5))
for j, label in enumerate(labels):
    values = K[:, 0, j]
    finite = np.isfinite(values)
    ax.plot(np.asarray(m)[finite], values[finite], label=rf"$K_{{1,{j+1}}}$ {label}")
ax.set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel=r"$K_{1j}(s)$", title=r"First row of the five-channel $K$ matrix")
ax.set_ylim(-20, 20)
ax.legend(ncol=2)
plt.show()


## 4. Production vector and rescattered amplitude

The physical production amplitude is

\[
F=(I-iK\rho)^{-1}P.
\]

The scalar inserted in the $\pi\pi$ S-wave component is $F_1$.


In [ ]:
P = np.asarray(kmatrix.production_vector(m))
F = np.asarray(kmatrix.amplitude_vector(m))
F1 = F[:, 0]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].plot(np.asarray(m), F1.real)
axes[0, 0].set_ylabel(r"Re $F_1$")
axes[0, 1].plot(np.asarray(m), F1.imag)
axes[0, 1].set_ylabel(r"Im $F_1$")
axes[1, 0].plot(np.asarray(m), np.abs(F1)**2)
axes[1, 0].set_ylabel(r"$|F_1|^2$")
axes[1, 1].plot(np.asarray(m), np.unwrap(np.angle(F1)))
axes[1, 1].set_ylabel("phase [rad]")
for ax in axes.flat:
    ax.set_xlabel(r"$m_{\pi\pi}$ [GeV]")
fig.suptitle(r"Physical $\pi\pi$ S-wave from the K-matrix/P-vector")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5.5))
for j, label in enumerate(labels):
    ax.plot(np.asarray(m), np.abs(P[:, j]), label=label)
ax.set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel=r"$|P_j|$", title="Illustrative production-vector components")
ax.legend(ncol=2)
plt.show()


## 5. One-component $D^+\to(\pi\pi)_S\pi^+$ model

The `Resonance` mass and width below are placeholders required by the common `ResonanceContext`; the K-matrix itself uses its own fixed scattering pole table. Since this is an S-wave, the external Blatt–Weisskopf factors and angular term are unity. The two identical $\pi^+$ are symmetrized automatically.


In [ ]:
model = DecayModel(
    channel,
    [
        Resonance(
            "pipi_S_kmatrix",
            pair=(0, 1),
            coefficient=RealImag(1.0, 0.0),
            mass=1.0,
            width=0.0,
            spin=0,
            lineshape=kmatrix,
            resonance_radius=3.0,
            parent_radius=3.0,
        )
    ],
    normalization_resolution=700,
    normalization_boundary_resolution=20001,
)

grid = model.normalization_sample
intensity = np.asarray(model.intensity(grid.as_dict()))
weights = np.asarray(grid.weights) * intensity
print("grid points =", grid.size)
print("all grid intensities finite =", np.all(np.isfinite(intensity)))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
h = ax.hist2d(np.asarray(grid.s12), np.asarray(grid.s13), bins=120, weights=weights)
fig.colorbar(h[3], ax=ax, label=r"grid weight $\times |A|^2$")
ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]", title=r"$D^+\to(\pi\pi)_S\pi^+$ — K-matrix")
plt.show()

s_pm = np.concatenate([np.asarray(grid.s12), np.asarray(grid.s13)])
w_pm = np.concatenate([weights, weights])
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(s_pm, bins=140, weights=w_pm, histtype="step", linewidth=1.6)
ax.set(xlabel=r"$m^2(\pi^+\pi^-)$ [GeV$^2$]", ylabel="weighted grid intensity", title=r"K-matrix $s_{12}+s_{13}$ projection")
plt.show()


## 6. 100k-event toy MC

MC is used only to generate the candidate pool. Amplitude/component/PDF normalization remains deterministic through the Dalitz grid.


In [ ]:
N_POOL = 1_000_000
N_TOY = 100_000
pool = model.generate_phase_space(N_POOL, seed=4100)
pool_intensity = model.intensity(pool.as_dict())
target_weights = pool.weights * pool_intensity

print("finite pool intensity =", bool(jnp.all(jnp.isfinite(pool_intensity))))
print("finite target weights =", bool(jnp.all(jnp.isfinite(target_weights))))

toy = weighted_resample(
    jax.random.key(4101),
    pool,
    target_weights,
    N_TOY,
    replace=True,
)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
h = ax.hist2d(np.asarray(toy.s12), np.asarray(toy.s13), bins=100)
fig.colorbar(h[3], ax=ax, label="events")
ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]", title=r"K-matrix toy — 100k events")
plt.show()

s_toy = np.concatenate([np.asarray(toy.s12), np.asarray(toy.s13)])
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(s_toy, bins=140, histtype="step", linewidth=1.6)
ax.set(xlabel=r"$m^2(\pi^+\pi^-)$ [GeV$^2$]", ylabel="entries / bin", title=r"K-matrix toy $s_{12}+s_{13}$ projection")
plt.show()
